# Study of dihedral angles

## Importations and tool functions

### Importations

In [57]:
%pip install biopython
%pip install ./clustangles-1.0-py3.tar.gz
%pip install umap-learn


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Processing ./clustangles-1.0-py3.tar.gz


  Installing build dependencies ... -

 \

...

  Attempting uninstall: clustangles
    Found existing installation: clustangles 1.0
    Uninstalling clustangles-1.0:
      Successfully uninstalled clustangles-1.0



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [58]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy as sc
import seaborn as sns
import numpy as np
import Bio as bp
import warnings
import clustangles as angles
import sklearn.manifold as sm
import sklearn.metrics as sme
import random as rdm
import umap
import numba
import scipy.stats
import math

from sklearn.linear_model import LinearRegression
from sklearn.cluster import HDBSCAN
from Bio.PDB import PDBParser, PDBIO, calc_dihedral
from scipy.optimize import linear_sum_assignment
from sklearn.neighbors import LocalOutlierFactor
from joblib import Parallel, delayed
from sklearn.preprocessing import LabelEncoder
from warnings import warn


rdm.seed(888)

In [59]:
#%run npy2pdb.py trinuc-AAC-0.5.npy AAC-template.pdb

In [60]:
#%run npy2pdb.py trinuc-CAC-0.5.npy CAC-template.pdb

In [61]:
#!python npy2pdb.py trinuc-AAC-0.5.npy AAC-template.pdb > trinuc-AAC-0.5.pdb

In [62]:
#!python npy2pdb.py trinuc-CAC-0.5.npy CAC-template.pdb > trinuc-CAC-0.5.pdb

In [63]:
print(bp.__version__)

1.87


In [64]:
warnings.filterwarnings("ignore")

In [65]:
# for model in ARN_AAC_test:
#     for chain in model:
#         print("Chain:", chain.id)
#         for residue in chain:
#             print("Residue:", residue.resname, residue.id)
#             for atom in residue:
#                 print("  Atom:", atom.name, atom.coord)

In [66]:
def get_angle(res_a, name_a, res_b, name_b, res_c, name_c, res_d, name_d):
    
    try:
        a, b = res_a[name_a].get_vector(), res_b[name_b].get_vector()
        c, d = res_c[name_c].get_vector(), res_d[name_d].get_vector()

        angle = np.degrees(calc_dihedral(a, b, c, d))

        return angle % 360

    except KeyError:
        return None

def get_full_structure(structure):
    data_list = []

    for model in structure:
        for chain in model:
            residues = list(chain.get_residues())
            for i in range(len(residues)):
                res = residues[i]
                res_prev = residues[i-1] if i > 0 else None
                res_next = residues[i+1] if i < len(residues) - 1 else None
                r_full_name = res.get_resname().strip()
                base_letter = r_full_name[-1]
            
                row = {
                    "chain": chain.id,
                    "res_id": res.id[1],
                    "res_name": r_full_name,
                
                    # Dorsale
                    "alpha":   get_angle(res_prev, "O3'", res, "P", res, "O5'", res, "C5'") if res_prev else None,
                    "beta":    get_angle(res, "P", res, "O5'", res, "C5'", res, "C4'"),
                    "gamma":   get_angle(res, "O5'", res, "C5'", res, "C4'", res, "C3'"),
                    "delta":   get_angle(res, "C5'", res, "C4'", res, "C3'", res, "O3'"),
                    "epsilon": get_angle(res, "C4'", res, "C3'", res, "O3'", res_next, "P") if res_next else None,
                    "zeta":    get_angle(res, "C3'", res, "O3'", res_next, "P", res_next, "O5'") if res_next else None,

                    # Sucre et Base
                    "nu0":     get_angle(res, "C4'", res, "O4'", res, "C1'", res, "C2'"),
                    "nu1":     get_angle(res, "O4'", res, "C1'", res, "C2'", res, "C3'"),
                    "nu2":     get_angle(res, "C1'", res, "C2'", res, "C3'", res, "C4'"),
                    "nu3":     get_angle(res, "C2'", res, "C3'", res, "C4'", res, "O4'"),
                    "nu4":     get_angle(res, "C3'", res, "C4'", res, "O4'", res, "C1'"),
                    
                    # Pseudoangles
                    "eta":     get_angle(res_prev, "C4'", res, "P", res, "C4'", res_next, "P") if (res_prev and res_next) else None,
                    "theta":   get_angle(res, "P", res, "C4'", res_next, "P", res_next, "C4'") if res_next else None
                }

                # Chi spécifique Purines/Pyrimidines
                if base_letter in ['A', 'G']:
                    row["chi"] = get_angle(res, "O4'", res, "C1'", res, "N9", res, "C4")
                elif base_letter in ['C', 'U']:
                    row["chi"] = get_angle(res, "O4'", res, "C1'", res, "N1", res, "C2")
                else:
                    row["chi"] = None

                data_list.append(row)

    # Création du DataFrame
    df_all_angles = pd.DataFrame(data_list)

    return(df_all_angles)

In [67]:
method_list = ["pseudo1", "pseudo2", "GeoPCA", "t-SNE", "UMAP"]
sources_list = ["all_6angles", "pseudo1", "pseudo2", "GeoPCA", "t-SNE", "UMAP"]
restricted_list = ["all_6angles", "pseudo1", "pseudo2", "t-SNE", "UMAP"]
bases_list = ["AAA", "AAC", "CAC", "CAA", "CCC", "CCA", "ACC", "ACA"]
conformations = ["C3' endo", "C4' exo", "O4' endo", "C1' exo", "C2' endo", "C3' exo", "C4' endo", "O4' exo", "C1' endo", "C2' exo"]

In [68]:
data_path = "/home/onyxia/work/Data/"
euclidian_path = "/home/onyxia/work/Data/3d_representations/"
distmat_path = "/home/onyxia/work/Distance Matrices/"
GeoPCA_fits_path = "/home/onyxia/work/GeoPCA_fits/"
TSNE_fits_path = "/home/onyxia/work/TSNE_fits/"
UMAP_fits_path = "/home/onyxia/work/UMAP_fits/"
HDBSCAN_fits_path = "/home/onyxia/work/HDBSCAN_fits/"
template_path = "/home/onyxia/work/Template/"

In [69]:
# parser = PDBParser(QUIET=True)

# structures = {}

# for base in bases_list :
#     template = f"{base}-template"
#     template_pdb = f"{template_path}{base}-template.pdb"
#     source_file = f"{euclidian_path}trinuc-{base}-0.5.pdb"
#     source_file_npy = f"{euclidian_path}trinuc-{base}-0.5.npy"

#     !python npy2pdb.py {source_file_npy} {template_pdb} > {source_file}
    
#     structures[base] = parser.get_structure(template, source_file)
    
#     file_name = f"angles_RNA_{base}.csv"
#     get_full_structure(structures[base]).to_csv(file_name, index = False)

In [70]:
dfs = {}

for base in bases_list:
    dfs[base] = pd.read_csv(f"{data_path}angles_RNA_{base}.csv")

df_all_angles = pd.concat(dfs, axis=1)

print(df_all_angles["AAA"]["alpha"])

0               NaN
1        305.490140
2        304.697410
3               NaN
4        146.988674
            ...    
18061    160.606135
18062    286.500302
18063           NaN
18064    144.721439
18065     62.626275
Name: alpha, Length: 18066, dtype: float64


In [71]:
df_all_angles["AAA"].describe()

,res_id,alpha,beta,gamma,delta,epsilon,zeta,nu0,nu1,nu2,nu3,nu4,eta,theta,chi
count,18066.000000,12044.000000,18066.000000,18066.000000,18066.000000,12044.000000,12044.000000,18066.000000,18066.000000,18066.000000,18066.000000,18066.000000,6022.000000,12044.000000,18066.000000
mean,2.000000,230.452941,174.845247,105.447378,93.173335,222.273500,250.444553,126.335010,282.358995,85.118928,275.077156,59.656076,170.937337,205.140479,198.921471
std,0.816519,87.800823,40.607273,73.106225,24.185263,34.236877,66.764142,163.389307,114.434279,109.138661,112.531055,109.545490,47.199056,51.546907,42.110130
min,1.000000,0.090459,40.772702,0.056789,12.163122,2.482037,0.632799,0.002840,0.196476,0.791058,0.156261,0.009875,0.725374,0.281131,0.152614
25%,1.000000,164.267416,152.930672,53.258921,80.664691,206.657167,236.810451,3.515364,328.408066,34.715927,321.220634,17.718702,156.804286,185.956224,183.868474
50%,2.000000,276.548482,172.543634,66.301911,82.891407,221.339775,276.182344,7.738748,333.956399,36.628326,324.336893,20.583588,169.116276,213.680739,194.703603
75%,3.000000,298.354133,199.304298,163.406659,88.167651,242.554031,290.469953,339.445401,336.660841,40.048217,326.146640,24.194042,182.290915,231.907046,211.816619
max,3.000000,359.983018,333.090798,359.897920,358.832641,353.445248,359.282497,359.996971,359.977408,359.312281,359.818321,359.999457,359.618143,359.231370,359.302816


In [72]:
mid_angles = {}
first_angles = {}
last_angles = {}

cols_to_drop = ["res_id", "res_name"]

for base in bases_list:
    
    mid_angles[base] = dfs[base][dfs[base]["res_id"].isin([2])].drop(columns=cols_to_drop)
    first_angles[base] = dfs[base][dfs[base]["res_id"].isin([1])].drop(columns=cols_to_drop)
    last_angles[base] = dfs[base][dfs[base]["res_id"].isin([3])].drop(columns=cols_to_drop)

mid_angles["AAA"]

,chain,alpha,beta,gamma,delta,epsilon,zeta,nu0,nu1,nu2,nu3,nu4,eta,theta,chi
1,B,305.490140,162.730043,52.603131,78.995842,214.526967,278.473410,0.070455,336.709644,36.383089,322.671813,23.477230,168.948446,209.995984,189.233622
4,B,146.988674,193.708737,179.793685,81.382929,225.230787,283.202503,8.016439,330.418764,38.904281,324.713641,17.215873,159.669284,195.684487,179.675616
7,B,301.320403,163.375457,50.046480,75.955382,223.673354,292.092174,4.371440,330.290735,42.360514,319.319860,22.886112,163.017130,223.839931,192.378723
10,B,295.578929,176.137498,64.087720,81.199826,209.114531,292.686375,358.939293,337.072121,36.751825,321.620907,24.848753,174.891792,219.747934,184.549837
13,B,301.868572,168.989858,52.243122,80.262432,210.286872,280.323108,0.935298,336.664007,35.627748,323.916667,22.172684,169.729129,208.902042,193.628678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18052,B,359.243083,115.818572,66.577611,83.483783,186.120445,256.786693,7.027008,332.352175,36.700126,326.430947,16.744271,185.351610,221.568289,297.780053
18055,B,143.786438,216.118476,186.337421,148.069144,208.983270,216.610609,338.920824,36.099064,323.462945,25.241961,357.198641,190.847730,170.360477,291.694825
18058,B,40.453015,250.096048,248.807834,147.528139,263.262144,71.416625,339.036145,35.579300,324.210889,24.606173,357.511716,230.553841,16.753732,50.805860
18061,B,160.606135,190.562729,173.702897,91.886990,229.348767,292.482836,17.771124,325.291458,37.452448,331.822386,6.719794,219.537182,209.696504,185.924817


In [73]:
mid_6angles = {}
first_6angles = {}
last_6angles = {}
mid_nu = {}
first_nu = {}
last_nu = {}

for base in bases_list:

    mid_6angles[base] = mid_angles[base][["alpha", "beta", "gamma", "delta", "epsilon", "zeta"]]
    first_6angles[base] = first_angles[base][["alpha", "beta", "gamma", "delta", "epsilon", "zeta"]]
    last_6angles[base] = last_angles[base][["alpha", "beta", "gamma", "delta", "epsilon", "zeta"]]
    
    mid_nu[base] = mid_angles[base][["nu0", "nu1", "nu2", "nu3", "nu4"]]
    first_nu[base] = first_angles[base][["nu0", "nu1", "nu2", "nu3", "nu4"]]
    last_nu[base] = last_angles[base][["nu0", "nu1", "nu2", "nu3", "nu4"]]

In [74]:
for base in bases_list:
    print(f"{base} usable size :{mid_6angles[base].shape[0]} | total size :{df_all_angles[base].shape[0]}")

AAA usable size :6022 | total size :18066
AAC usable size :5342 | total size :18066
CAC usable size :3528 | total size :18066
CAA usable size :5253 | total size :18066
CCC usable size :5210 | total size :18066
CCA usable size :4622 | total size :18066
ACC usable size :4787 | total size :18066
ACA usable size :4044 | total size :18066


### Tool functions

In [75]:
#function torian distance

@numba.njit()
def tore_dist(x, y, angles_mesure = "d", return_grad=False):
    if angles_mesure == "d" :
        angle_value = 360
    if angles_mesure == "rad" :
        angle_value = 2*np.pi
   
    x, y = np.asarray(x), np.asarray(y)
    
    diff = x - y
    half_val = angle_value / 2
    short_diff = (diff + half_val) % angle_value - half_val
    
    dist_sq = np.sum(short_diff**2)
    dist = np.sqrt(dist_sq)
        
    if not return_grad:
       return dist

    if dist < 1e-10:
        grad = np.zeros_like(x)
    else:
        grad = short_diff / dist
    
    return dist, grad

In [76]:
#function torian distance for the UMAP formalism
@numba.njit()
def tore_dist_umap(x, y):
    angle_value = 360.0
    half_val = 180.0
    
    dist_sq = 0.0
    for i in range(len(x)):
        diff = x[i] - y[i]
        short_diff = (diff + half_val) % angle_value - half_val
        dist_sq += short_diff**2
        
    return np.sqrt(dist_sq)

In [77]:
def P_fun(thetas, tau_out = False):

    A = B = 0
    
    for i in range(5):
        alpha_deg = 144 * i
        alpha_rad = math.radians(alpha_deg)
        A += thetas.iloc[i] * math.cos(alpha_rad)
        B += thetas.iloc[i] * math.sin(alpha_rad)

    A = (2/5) * A
    B = (-2/5) * B

    tau_m = math.sqrt(A**2 + B**2)
    P_rad = math.atan2(B, A)

    P_deg = math.degrees(P_rad)
    
    if P_deg < 0:
        P_deg += 360
    
    if tau_out:
        return(P_deg, tau_m)
        
    return(P_deg)

In [78]:
def get_pucker_conf(P_theta):
    index = int(np.floor(P_theta/36))
    
    return(conformations[index])

In [ ]:
def order_fun(ang1, ang2, value = True):
    mod = (ang1 - ang2 + 360) % 360
    out_value = mod - 180

    final = 'anti-clockwise'
    if(out_value > 0):
        final = 'clockwise'
    if(out_value == 0):
        final = 'no rotation'
    
    if(value == True):
        return(out_value, final)

    return(final)

In [ ]:
order_fun(68,250)

## Representation for the middle nucleotides

### Pairplot representation

In [ ]:
P_thetas = {}

for base in bases_list :

    P_thetas[base] = mid_angles[base][["nu0", "nu1", "nu2", "nu3", "nu4"]].apply(
        lambda row: P_fun(row), axis=1)

P_thetas["AAA"]

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, base in enumerate(bases_list):
    sns.histplot(P_thetas[base],
                 ax = axes[i])
    axes[i].set_title(base)

plt.suptitle(r"$P_{\theta}$ distributions",
             size = 20)
plt.show()

In [ ]:
conformations_clusters = {}

for base in bases_list :
    conformations_clusters[base] = P_thetas[base].apply(get_pucker_conf)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

axes = axes.flatten()
for i, base in enumerate(bases_list) :
    sns.countplot(x = conformations_clusters[base],
                  ax = axes[i],
                  order = conformations)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_title(base)
    axes[i].set_xlabel("")

plt.suptitle("Conformation clusters", size = 20)
plt.show()

In [ ]:
group_A = ["AAA", "AAC", "CAA", "CAC"]
group_C = ["CCC", "CCA", "ACC", "ACA"]

dfs_combined = []
for base in bases_list:
    df_temp = mid_angles[base].copy()
    df_temp["base"] = base
    df_temp["group"] = "A" if base in group_A else "C"
    df_temp["conformation"] = conformations_clusters[base].values
    dfs_combined.append(df_temp)

df_combined = pd.concat(dfs_combined)

fig, ax = plt.subplots(figsize = (10, 5))

sns.countplot(data = df_combined,
              x = "conformation",
              hue = "group",
              ax = ax,
              palette = ['#3C2EFF', '#FF7676'])

ax.tick_params(axis='x',
               rotation=45)
ax.set_xlabel("")
plt.tight_layout()
plt.title("Conformation clusters", size = 20)
plt.show()

In [ ]:
df_pct = (df_combined.groupby(["group", "conformation"])
                     .size()
                     .groupby(level=0, group_keys=False)
                     .apply(lambda x: 100 * x / x.sum())
                     .reset_index(name="percentage"))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_pct,
            x="conformation",
            y="percentage",
            hue="group",
            ax=ax,
            palette=['#3C2EFF', '#FF7676'])

ax.tick_params(axis='x', rotation=45)
ax.set_xlabel("")
ax.set_ylabel("%")
plt.tight_layout()
plt.title("Conformation clusters", size=20)
plt.show()

In [ ]:
sns.pairplot(data = mid_6angles["AAC"], corner = True)

In [ ]:
sns.pairplot(data = mid_angles["AAA"][["nu0", "nu1", "nu2", "nu3", "nu4"]], corner = True)
plt.title(r"$\nu$ distributions for AAA")

In [ ]:
sns.pairplot(data = mid_angles["CCC"][["nu0", "nu1", "nu2", "nu3", "nu4"]], corner = True)
plt.title(r"$\nu$ distributions for CCC")

### Pseudoangles

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (24,15))
axes = axes.flatten()

for i, base in enumerate(bases_list):
    axes[i].scatter(x = mid_angles[base]["eta"], y = mid_angles[base]["theta"], c = "black", s = 3)
    axes[i].set_xlabel(r"$\eta$")
    axes[i].set_ylabel(r"$\theta$")
    axes[i].set_title(base)
    
plt.suptitle(r"$\eta$ - $\theta$ graph", size = 20)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (24,15))
axes = axes.flatten()

for i, base in enumerate(bases_list):
    axes[i].scatter(x = mid_angles[base]["alpha"], y = mid_angles[base]["zeta"], c = "black", s = 3)
    axes[i].set_xlabel(r"$\alpha$")
    axes[i].set_ylabel(r"$\zeta$")
    axes[i].set_title(base)
    
plt.suptitle(r"$\alpha$ - $\zeta$ graph", size = 20)

### GeoPCA

In [ ]:
# geo_6angles = {}
# g = {}

# for base in bases_list:
#     csv_name = f"geo_6angles_{base}.csv"
#     #mid_6angles[base].to_csv(csv_name, index = False, header = False)
#     geo_6angles[base] = angles.Angles()
#     geo_6angles[base].read_csv(csv_name, units = "degrees")
#     geo = angles.geopca(geo_6angles[base].unitsphere())
#     g[base] = geo.project(geo_6angles[base].unitsphere())

In [ ]:
geopca_fit = {}

for base in bases_list:
    csv_name = f"{GeoPCA_fits_path}GeoPCA_fit_{base}.npy"
    #np.save(csv_name, g[base].dataset)
    geopca_fit[base] = np.load(csv_name).T

In [ ]:
dir(geopca_fit)

In [ ]:
geopca_fit.update()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (24,10))
axes = axes.flatten()

for i, base in enumerate(bases_list):
    axes[i].scatter(x = geopca_fit[base][0], y = geopca_fit[base][1], c = "black", s = 3)
    axes[i].set_title(base)
    
plt.suptitle(r"GeoPCA projection", size = 20)
plt.show()

### t-SNE

In [ ]:
TSNE_fit = {}

for base in bases_list :

    npy_name = f"{TSNE_fits_path}TSNE_fit_{base}.npy"
    #TSNE_fit[base] = sm.TSNE(metric=tore_dist).fit_transform(mid_6angles[base])
    #np.save(npy_name, TSNE_fit[base]) 
    TSNE_fit[base] = np.load(npy_name)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (20,10))
axes = axes.flatten()

for i, base in enumerate(bases_list):
    axes[i].scatter(x = TSNE_fit[base][:,0], y = TSNE_fit[base][:,1], c = "black", s = 3)
    axes[i].set_title(base)
    
plt.suptitle(r"t-SNE projection", size = 20)

### UMAP

In [ ]:
UMAP_fit = {}

for base in bases_list :

    npy_name = f"{UMAP_fits_path}UMAP_fit_{base}.npy"
    #UMAP_fit[base] = umap.UMAP(metric=tore_dist_umap).fit_transform(mid_6angles[base])
    #np.save(npy_name, UMAP_fit[base]) 
    UMAP_fit[base] = np.load(npy_name)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (20,10))
axes = axes.flatten()

for i, base in enumerate(bases_list):
    axes[i].scatter(x = UMAP_fit[base][:,0], y = UMAP_fit[base][:,1], c = "black", s = 3)
    axes[i].set_title(base)
    
plt.suptitle(r"UMAP projection", size = 20)

## Clustering

### Cluster calculation

In [ ]:
pseudo = {base: mid_angles[base][["eta", "theta"]] for base in bases_list}
pseudo_bis = {base: mid_angles[base][["alpha", "zeta"]] for base in bases_list}

In [ ]:
def fit_hdbscan(method, base, mcs, data):
    clusterer = HDBSCAN(min_cluster_size=mcs, metric=tore_dist, copy=True)
    result = clusterer.fit(data[base])
    print([method, base, mcs])
    return (method, base, mcs, result)

# results = Parallel(n_jobs=-1, backend='threading')(
#     delayed(fit_hdbscan)(method, base, mcs, data)
#     for method, data in sources_list.items()
#     for base in bases_list
#     for mcs in params_list
# )

# hdbscan_stock = {}
# for method, base, mcs, clusterer in results:
#     hdbscan_stock.setdefault(method, {}).setdefault(base, {})[mcs] = clusterer

In [ ]:
#np.save("hdbscan_stock_full.npy", hdbscan_stock)       
hdbscan_stock = np.load(f"{HDBSCAN_fits_path}hdbscan_stock_full.npy", allow_pickle=True).item()

In [ ]:
for base in bases_list:
    print(base)
    print(len(hdbscan_stock["pseudo1"][base][100].labels_))
    print(np.unique(hdbscan_stock["pseudo1"][base][100].labels_))

In [ ]:
test_pseudo1_AAA_100 = HDBSCAN(min_cluster_size=100, metric=tore_dist_umap).fit(pseudo["AAA"])

In [ ]:
np.unique(test_pseudo1_AAA_100.labels_)

### HDBSCAN comparison

In [ ]:
def clean_confusion_matrix(cm):
    cm = cm[1:, 1:]
    row_mask = ~np.all(cm == 0, axis=1)
    col_mask = ~np.all(cm == 0, axis=0)
    return cm[row_mask][:, col_mask]

In [ ]:
def reorder_cols_for_diagonal(cm):

        n_rows, n_cols = cm.shape
        min_dim = min(n_rows, n_cols)
        row_ind, col_ind = linear_sum_assignment(-cm[:min_dim, :])
        
        assigned_cols = set(col_ind)
        remaining_cols = [c for c in range(n_cols) if c not in assigned_cols]
        
        full_col_order = list(col_ind) + remaining_cols
        return cm[:, full_col_order]

In [ ]:
def compars_HDBSCAN(method, base, represented_data, title, xlabel, ylabel):

    if(method not in restricted_list):
        raise ValueError(f"method has to be one of the following : {restricted_list}")

    represented_data = pd.DataFrame(represented_data).reset_index(drop=True)

    # Confusion matrix creation
    def make_cm(mcs):

        cm = sme.confusion_matrix(
            hdbscan_stock[method][base][mcs].labels_,
            hdbscan_stock['all_6angles'][base][mcs].labels_
        )
        cm_cleaned = clean_confusion_matrix(cm).astype(float)
        cm_heat = reorder_cols_for_diagonal(cm_cleaned)
        cm_heat[cm_heat == 0] = np.nan

        return cm_heat



    cm_heat_30  = make_cm(30)
    cm_heat_50  = make_cm(50)
    cm_heat_100 = make_cm(100)

    # Graph creation
    fig, axes = plt.subplots(3, 4, figsize=(24, 15))

    for i, (mcs, cm_heat, clusterer, T6_clusterer) in enumerate(zip(
        [30, 50, 100],
        [cm_heat_30, cm_heat_50, cm_heat_100],
        [hdbscan_stock[method][base][30],      hdbscan_stock[method][base][50],      hdbscan_stock[method][base][100]],
        [hdbscan_stock['all_6angles'][base][30], hdbscan_stock['all_6angles'][base][50], hdbscan_stock['all_6angles'][base][100]]
    )):

        labels    = clusterer.labels_
        T6_labels = T6_clusterer.labels_
        
        noise_mask    = labels == -1
        T6_noise_mask = T6_labels == -1
        has_clusters  = len(np.unique(labels[labels != -1])) > 0
        ARI = round(sme.adjusted_rand_score(labels, T6_labels), 3)

        # Info panel
        axes[i, 0].axis('off')
        axes[i, 0].text(0.5, 0.5, f'min_cluster_size={mcs}', fontsize=14, ha='center', va='center')
        axes[i, 0].text(0.5, 0.3, f'ARI={ARI}', fontsize=14, ha='center')
       
        # Scatter plot method
        axes[i, 1].scatter(represented_data.iloc[:, 0][noise_mask],  represented_data.iloc[:, 1][noise_mask],
                           s=3, c='lightgrey', alpha=.3)
        if has_clusters:
            axes[i, 1].scatter(represented_data.iloc[:, 0][~noise_mask], represented_data.iloc[:, 1][~noise_mask],
                               s=3, c=labels[~noise_mask], cmap='tab20b', alpha=.5)

        axes[i, 1].set_title(title)
        if not (isinstance(xlabel, bool) and isinstance(ylabel, bool)):
            axes[i, 1].set_xlabel(xlabel)
            axes[i, 1].set_ylabel(ylabel)        

        # Heatmap
        if cm_heat.size == 0 or not has_clusters:
            axes[i, 2].text(0.5, 0.5, 'No clusters found', ha='center', va='center', fontsize=12)
            axes[i, 2].axis('off')

        else:
            sns.heatmap(cm_heat, ax=axes[i, 2], cmap='rocket_r', annot=True,
                        cbar_kws={'label': 'Occupancy'})
        axes[i, 2].set_title('Confusion matrix\n')
        axes[i, 2].set_xlabel(r'$\mathbb{T}^6$ clusters')
        axes[i, 2].set_ylabel(title)

        # Scatter plot T6
        axes[i, 3].scatter(represented_data.iloc[:, 0][T6_noise_mask],  represented_data.iloc[:, 1][T6_noise_mask],
                           s=3, c='lightgrey', alpha=.3)
        axes[i, 3].scatter(represented_data.iloc[:, 0][~T6_noise_mask], represented_data.iloc[:, 1][~T6_noise_mask],
                           s=3, c=T6_labels[~T6_noise_mask], cmap='tab20b', alpha=.5)
        axes[i, 3].set_title(r'$\mathbb{T}^6$ clusters')
        
        if not (isinstance(xlabel, bool) and isinstance(ylabel, bool)):
            axes[i, 3].set_xlabel(xlabel)
            axes[i, 3].set_ylabel(ylabel)           

    fig.suptitle(f"Cluster preservation — {method} | base={base}", fontsize=20)
    plt.tight_layout()

    plt.show()

In [ ]:
compars_HDBSCAN(method = 'all_6angles', base = "CAC", represented_data = mid_6angles["CAC"], title = r'$\eta$ - $\theta$ clusters', xlabel = r'$\eta$', ylabel = r'$\theta$')

In [ ]:
compars_HDBSCAN(method = 'UMAP', base = "CAC", represented_data = UMAP_fit["CAC"], title = r'UMAP clusters', xlabel = False, ylabel = False)

In [ ]:
compars_HDBSCAN(method = 'pseudo1', base = "ACC", represented_data = pseudo["ACC"], title = r'$\eta$-$\theta$ clusters', xlabel = r'$\eta$', ylabel = r'$\theta$')

### Pucker conformations comparison

In [ ]:
PC_stock = {}

for base in bases_list:
    P_values = mid_nu[base].apply(lambda row: P_fun(row), axis=1)
    PC_stock[base] = P_values.apply(get_pucker_conf)

In [ ]:
def heat_conformations(method, mcs = 30):

    fig, axes = plt.subplots(2, 4, figsize = (28, 15))

    axes = axes.flatten()

    for i, base in enumerate(bases_list):
        le = LabelEncoder()
        PC_encoded = le.fit_transform(PC_stock[base].reset_index(drop=True))
    
        cm = sme.confusion_matrix(PC_encoded,  hdbscan_stock[method][base][mcs].labels_)
        cm_cleaned = clean_confusion_matrix(cm).astype(float)
        cm_heat = reorder_cols_for_diagonal(cm_cleaned)
        cm_heat[cm_heat == 0] = np.nan

        ARI = round(sme.adjusted_rand_score(PC_encoded, hdbscan_stock[method][base][mcs].labels_), 3)

    
        sns.heatmap(cm_heat, ax=axes[i], cmap='rocket_r', annot=True,
                    cbar_kws={'label': 'Occupancy'},
                    yticklabels=le.classes_)  
    
        axes[i].set_title(f"base : {base} | ARI = {ARI}")
        axes[i].set_xlabel(fr'{method} clusters')

        if(method == "all_6angles"):
            axes[i].set_xlabel(r'$\mathbb{T}^6$ clusters')
        axes[i].set_ylabel('Pucker conformation')

    fig.suptitle(fr"Pucker conformation vs {method} clusters (mcs= {mcs})", fontsize=20)    
    if(method == "all_6angles"):
        fig.suptitle(r"Pucker conformation vs $\mathbb{T}^6$" + f"clusters (mcs={mcs})", fontsize=20)
        
    plt.tight_layout()
    plt.show()

In [ ]:
heat_conformations(method = 'all_6angles',mcs = 30)

## Distances comparisons

### Distances calculations

In [ ]:
def dist_calc(method, base, data):
    distance = sme.pairwise_distances(X = data[base], metric = tore_dist, n_jobs = 1)
    print([method, base])
    return (method, base, distance)

# results = Parallel(n_jobs=8, backend='threading')(
#     delayed(dist_calc)(method, base, data)
#     for method, data in sources_list.items()
#     for base in bases_list
# )

# distances_stock = {}
# for method, base, distance in results:
#     distances_stock.setdefault(method, {})[base] = distance

In [ ]:
#np.save("distances_stock_full.npy", distances_stock)
distances_stock = np.load(f"{distmat_path}distances_stock_full.npy", allow_pickle=True).item()

In [ ]:
array_trig_stock = {}

for method in distances_stock:
    array_trig_stock[method] = {}
    for base in bases_list:
        idx = np.triu_indices_from(distances_stock[method][base], k=1)
        array_trig_stock[method][base] = distances_stock[method][base][idx]

In [ ]:
array_trig_stock

In [ ]:
compars_dist = {}

for base in bases_list:
    # random_draw basé sur la taille de T6 (référence)
    random_draw = rdm.sample(range(array_trig_stock['all_6angles'][base].size), k=2000)
    
    compars_dist[base] = pd.DataFrame({
        method: array_trig_stock[method][base][random_draw]
        for method in array_trig_stock
    })

In [ ]:
compars_dist

### Independance and correlation tests

In [ ]:
pearson_stock = {}

for base in bases_list:
    pearson_stock[base] = {}
    for method in array_trig_stock:
        if method == 'all_6angles':
            continue
        pearson_stock[base][method] = scipy.stats.pearsonr(
            compars_dist[base]['all_6angles'],
            compars_dist[base][method]
        )

In [ ]:
chi2_stock = {}

for base in bases_list:
    chi2_stock[base] = {}
    for method in array_trig_stock:
        if method == 'all_6angles':
            continue
        
        table = [compars_dist[base]['all_6angles'],
                 compars_dist[base][method]]
        
        result = scipy.stats.chi2_contingency(table)
        chi2_stock[base][method] = result[0:2]  # (statistic, pvalue)

In [ ]:
for method in restricted_list[1:len(restricted_list)]:
    print()
    print(method)
    print(f"     | pear | chi2")
    for base in bases_list:
        print(f"{base} | {pearson_stock[base][method].pvalue:.3g} | {chi2_stock[base][method][1]:.3g}")     

### Representation

In [ ]:
def get_extremes(data_x, data_y, k=5):
    points = np.column_stack((data_x, data_y))
    clf = LocalOutlierFactor(n_neighbors=20)
    clf.fit_predict(points)
    return np.argsort(clf.negative_outlier_factor_)[:k]

In [ ]:
extreme_stock = {}

for base in bases_list:
    indices_per_method = []
    for method in array_trig_stock:
        if method == 'all_6angles':
            continue
        indices = get_extremes(
            compars_dist[base]['all_6angles'],
            compars_dist[base][method]
        )
        indices_per_method.append(indices)
    
    extreme_stock[base] = np.unique(np.concatenate(indices_per_method))

In [ ]:
model_stock = {}

for base in bases_list:
    model_stock[base] = {}
    for method in array_trig_stock:
        if method == 'all_6angles':
            continue
        model_stock[base][method] = LinearRegression().fit(
            compars_dist[base][['all_6angles']],
            compars_dist[base][[method]]
        )

In [ ]:
def plot_scatter_dist_compars(ax, base, y_col, model, indices, ylabel, pvalue, xlabel=r"$\mathbb{T}^6$ pairwise distance"):
    sns.scatterplot(data=compars_dist[base], x='all_6angles', y=y_col, ax=ax)
    xlims = ax.get_xlim()
    ylims = ax.get_ylim()
    
    # Droite y=x
    common_min = min(xlims[0], ylims[0])
    common_max = max(xlims[1], ylims[1])
    ax.plot([common_min, common_max], [common_min, common_max],
            color='black', linestyle='-', alpha=0.7, zorder=1, label='y=x')
    
    # Droite OLS
    X_ = pd.DataFrame(np.linspace(xlims[0], xlims[1], 100).reshape(-1, 1), columns=['all_6angles'])
    ax.plot(X_, model.predict(X_), color='red', linestyle='--', alpha=0.7, label='OLS')
    
    if type(indices) is not bool:
        ax.scatter(data=compars_dist[base].loc[indices], x='all_6angles', y=y_col,
                   color='red', label='extreme values')
    ax.set_xlim(xlims)
    ax.set_ylim(ylims)
    ax.legend(loc='upper left')
    ax.text(.03, .73, pvalue, transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='black'))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

plot_config = {
    'pseudo1': r"$\eta$ - $\theta$ pairwise distance",
    'pseudo2': r"$\alpha$ - $\zeta$ pairwise distance",
    't-SNE':   r"t-SNE pairwise distance",
    'UMAP':    r"UMAP pairwise distance",
}

def plot_distance_conservation(base):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    
    for ax, (method, ylabel) in zip(axes.flatten(), plot_config.items()):
        plot_scatter_dist_compars(
            ax      = ax,
            base    = base,
            y_col   = method,
            model   = model_stock[base][method],
            indices = extreme_stock[base],
            ylabel  = ylabel,
            pvalue  = f"pvalue = {pearson_stock[base][method].pvalue:.3g}"
        )
    
    fig.suptitle(f"Distance conservation — base : {base}", fontsize=20)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_distance_conservation(base='CAC')

## Neighborhood comparisons

### On every points

In [ ]:
k_list = [30, 50, 100]

KNN_stock = {}
compars_jaccard_stock = {}
jaccard_scores_stock = {}
mean_jaccard_stock = {}

for k in k_list:
    KNN_stock[k] = {}
    compars_jaccard_stock[k] = {}
    jaccard_scores_stock[k] = {}
    mean_jaccard_stock[k] = {}   

    for method in distances_stock:
        KNN_stock[k][method] = {}
        
        for base in bases_list:
            KNN_stock[k][method][base] = np.argsort(
                distances_stock[method][base], axis=1
            )[:, 1:k+1]

    for base in bases_list:
        compars_jaccard_stock[k][base] = pd.DataFrame({
            method: list(KNN_stock[k][method][base])
            for method in distances_stock
        })
        
        jaccard_scores_stock[k][base] = {}
        mean_jaccard_stock[k][base] = {}        

        for method in distances_stock:
            if method == 'all_6angles':
                continue
            jaccard_scores_stock[k][base][method] = compars_jaccard_stock[k][base].apply(
                lambda row, m=method: scipy.spatial.distance.jaccard(row['all_6angles'], row[m]),
                axis=1
            )
            
            mean_jaccard_stock[k][base][method] = np.mean(jaccard_scores_stock[k][base][method])

In [ ]:
def plot_hist_nei_compars(ax, data, xlabel, bins=25):
    
    sns.histplot(data=data, ax=ax, bins=bins)
    ax.set_xlabel(xlabel)

def plot_neighborhood_conservation(base, k):
    
    if(k not in k_list):
        raise ValueError(f"k has to be one of the following : {k_list}, given : {k}")
    if(base not in bases_list):
        raise ValueError(f"base has to be one of the following : {bases_list}, given : {base}")
        
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    
    for ax, (method, xlabel) in zip(axes.flatten(), plot_config.items()):
        plot_hist_nei_compars(
            ax     = ax,
            data   = jaccard_scores_stock[k][base][method],
            xlabel = xlabel
        )
    
    fig.suptitle(f"Neighborhood conservation — k={k} | base={base}", fontsize=20)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_neighborhood_conservation(base='ACA', k=100)

### By group of 1000 individuals

In [ ]:
n_sampling = 50

def neighbour_study(base, sample_size=1000, k_=100):
    
    range_size = distances_stock['all_6angles'][base].shape[0]
    point_sampling = rdm.sample(range(range_size), k=sample_size)
    
    def get_knn(dist_mat):
        out = []
        for i in point_sampling:
            row = dist_mat[i]
            neighbours = np.argpartition(row, k_+1)[:k_+1]
            out.append(neighbours[neighbours != i])
        return out
    
    KNN = pd.DataFrame({
        method: list(get_knn(distances_stock[method][base]))
        for method in distances_stock
    })
    
    return KNN

In [ ]:
import gc

def multiple_neighbour_study(base, sample_size=1000, k_=100):
    runs = []
    for i in range(n_sampling):
        df = neighbour_study(base=base, sample_size=sample_size, k_=k_)
        df['run'] = i + 1
        runs.append(df)
        gc.collect()
    return pd.concat(runs, ignore_index=True)

# Pour toutes les bases
multiple_neighbours_stock = {
    base: multiple_neighbour_study(base=base)
    for base in bases_list
}

In [ ]:
run_choice = 20

def compute_multiple_jaccard(base, run_choice):
    df_run = multiple_neighbours_stock[base].loc[
        multiple_neighbours_stock[base]['run'] == run_choice
    ]
    
    scores = {}
    mean_scores = {}
    
    for method in distances_stock:
        if method == 'all_6angles':
            continue
        scores[method] = df_run.apply(
            lambda row: scipy.spatial.distance.jaccard(row['all_6angles'], row[method]),
            axis=1
        )
        mean_scores[method] = np.mean(scores[method])
    
    return scores, mean_scores

# Pour toutes les bases
multiple_jaccard_stock = {}
mean_multiple_jaccard_stock = {}

for base in bases_list:
    multiple_jaccard_stock[base], mean_multiple_jaccard_stock[base] = compute_multiple_jaccard(
        base       = base,
        run_choice = run_choice
    )

In [ ]:
def plot_multiple_neighborhood_conservation(base, run_choice):
    scores, _ = compute_multiple_jaccard(base=base, run_choice=run_choice)
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    
    for ax, (method, xlabel) in zip(axes.flatten(), plot_config_jaccard.items()):
        plot_hist_nei_compars(
            ax     = ax,
            data   = scores[method],
            xlabel = xlabel
        )
    
    fig.suptitle(f"Neighborhood conservation for 1000 individuals for the run {run_choice} — base={base}", fontsize=20)
    plt.tight_layout()
    plt.show()

plot_config_jaccard = {
    'pseudo1': r"$\mathbb{T}^6$ vs $\eta$ - $\theta$",
    'pseudo2': r"$\mathbb{T}^6$ vs $\alpha$ - $\zeta$",
    't-SNE':   r"$\mathbb{T}^6$ vs t-SNE",
    'UMAP':    r"$\mathbb{T}^6$ vs UMAP",
}

In [ ]:
plot_multiple_neighborhood_conservation(base='CCC', run_choice=50)

### On the extreme values

In [ ]:
k_ex = 250
KNN_extreme_stock = {}

for method in distances_stock:
    KNN_extreme_stock[method] = {}
    for base in bases_list:
        KNN_extreme_stock[method][base] = np.argsort(
            distances_stock[method][base][extreme_stock[base], ],
            axis=1
        )[:, 1:k_ex+1]

In [ ]:
compars_jaccard_extreme_stock = {}
extreme_jaccard_scores_stock = {}

for base in bases_list:
    compars_jaccard_extreme_stock[base] = pd.DataFrame({
        method: list(KNN_extreme_stock[method][base])
        for method in KNN_extreme_stock
    })
    
    extreme_jaccard_scores_stock[base] = {}
    
    for method in distances_stock:
        if method == 'all_6angles':
            continue
        extreme_jaccard_scores_stock[base][method] = compars_jaccard_extreme_stock[base].apply(
            lambda row: scipy.spatial.distance.jaccard(row['all_6angles'], row[method]),
            axis=1
        )

In [ ]:
def ordored_nei(row1,row2):
    length = len(row1)
    out = 0
    for i in range(length):
        if (row1[i] != row2[i]) :
            out += 1 
    return(out/length)
    
ordered_extreme_stock = {}

for base in bases_list:
    ordered_extreme_stock[base] = {}
    
    for method in distances_stock:
        if method == 'all_6angles':
            continue
        ordered_extreme_stock[base][method] = compars_jaccard_extreme_stock[base].apply(
            lambda row: ordored_nei(row['all_6angles'], row[method]),
            axis=1
        )

In [ ]:
all_extreme_jaccard_stock = {}
all_ordered_extreme_stock = {}

for base in bases_list:
    all_extreme_jaccard_stock[base] = pd.concat(
        [extreme_jaccard_scores_stock[base][method] for method in extreme_jaccard_scores_stock[base]],
        axis=1
    )
    all_extreme_jaccard_stock[base].columns = list(extreme_jaccard_scores_stock[base].keys())
    
    all_ordered_extreme_stock[base] = pd.concat(
        [ordered_extreme_stock[base][method] for method in ordered_extreme_stock[base]],
        axis=1
    )
    all_ordered_extreme_stock[base].columns = list(ordered_extreme_stock[base].keys())

In [ ]:
xticklabels = [r'$\mathbb{T}^6$/P1', r'$\mathbb{T}^6$/P2', r'$\mathbb{T}^6$/t-SNE', r'$\mathbb{T}^6$/UMAP']

def plot_extreme_heatmaps(base):
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    sns.heatmap(all_extreme_jaccard_stock[base], annot=True, ax=axes[0],
                xticklabels=xticklabels,
                cbar_kws={'label': 'Jaccard Distance'})
    axes[0].set_title(f"Jaccard distance for the extreme values")
    
    sns.heatmap(all_ordered_extreme_stock[base], annot=True, fmt=".3g", ax=axes[1],
                xticklabels=xticklabels,
                cbar_kws={'label': 'Dissimilarity'}, cmap='rocket_r')
    axes[1].set_title(f"Ordered dissimilarity for the extreme values")

    plt.suptitle(f"Base = {base}", fontsize = 20)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_extreme_heatmaps(base='ACA')

## Cluster specifications

### Pairwise representations

#### HDBSCAN

In [ ]:
def pairwise_methods_HDBSCAN(data, method, base, mcs = 50):
    labels = hdbscan_stock[method][base][mcs].labels_
    df = data.copy().reset_index(drop=True)
    df['clusters'] = labels
    
    clusters = np.unique(labels)
    
    palette = {}
    tab10 = plt.get_cmap('tab10')
    color_idx = 0
    for c in clusters:
        if c == -1:
            palette[c] = 'lightgrey'
        else:
            palette[c] = tab10(color_idx + 1)
            color_idx += 1

    g = sns.pairplot(data=df, corner=True, hue='clusters', palette = palette)

    if method == 'all_6angles':
        title = r'$\mathbb{T}^6$' + f' with {base} clusters'
    if method == 'pseudo1':
        title = r'$\eta$ - $\theta$' + f' with {base} clusters'
    if method == 'pseudo2':
        title = r'$\alpha$ - $\zeta$' + f' with {base} clusters'
    else:
        title = f"{method} with {base} clusters"
        
    g.legend.set_title(title)
    g.legend.set_bbox_to_anchor((.85, .37))

In [ ]:
pairwise_methods_HDBSCAN(data = mid_6angles["AAA"], method = "t-SNE", base = "AAA")

In [ ]:
pairwise_methods_HDBSCAN(data = mid_6angles["AAA"], method = "UMAP", base = "AAA")

#### Pucker confirmation

In [ ]:
def pairwise_methods_PC(data, base):
    labels = PC_stock[base].reset_index(drop=True)
    df = data.copy().reset_index(drop=True)
    df['clusters'] = labels
    
    clusters = np.unique(labels)
    
    palette = {}
    tab10 = plt.get_cmap('tab10')
    color_idx = 0
    for c in clusters:
        palette[c] = tab10(color_idx + 1)
        color_idx += 1

    g = sns.pairplot(data=df, corner=True, hue='clusters', palette=palette)
    g.legend.set_title(f'Pucker conformation with {base}')
    g.legend.set_bbox_to_anchor((.85, .37))
    plt.show()

In [ ]:
pairwise_methods_PC(data = mid_6angles['AAA'], base = 'AAA')

In [ ]:
pairwise_methods_PC(data = mid_6angles['CCC'], base = 'CCC')

### Projections

#### On the biggest cluster of $\mathbb{T}^6$

In [ ]:
def get_biggest_cluster_HDBSCAN(clust_stock, method, base, mcs=30):
    
    labels = hdbscan_stock[method][base][mcs].labels_
    unique, counts = np.unique(labels[labels != -1], return_counts=True)
    largest_class = unique[np.argmax(counts)]
    mask = hdbscan_stock[method][base][mcs].labels_ == largest_class
    cluster = clust_stock.reset_index(drop=True)[mask] 
    
    return cluster

In [ ]:
T6_biggest_HDBSCAN_cluster = get_biggest_cluster_HDBSCAN(clust_stock = mid_6angles["AAA"], method = "all_6angles", base = "AAA")

In [ ]:
T6_biggest_HDBSCAN_cluster_TSNE = sm.TSNE(metric=tore_dist_umap).fit_transform(T6_biggest_HDBSCAN_cluster)

In [ ]:
T6_biggest_HDBSCAN_cluster_UMAP = umap.UMAP(metric=tore_dist_umap).fit_transform(T6_biggest_HDBSCAN_cluster)

In [ ]:
T6_biggest_HDBSCAN_cluster_TSNE_clustering = HDBSCAN(min_cluster_size=30, metric=tore_dist_umap, copy=True).fit(T6_biggest_HDBSCAN_cluster_TSNE)

In [ ]:
T6_biggest_HDBSCAN_cluster_UMAP_clustering = HDBSCAN(min_cluster_size=30, metric=tore_dist_umap, copy=True).fit(T6_biggest_HDBSCAN_cluster_UMAP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

for ax, projection, clustering, proj_name in zip(
    [axes[0], axes[1]],
    [T6_biggest_HDBSCAN_cluster_TSNE, T6_biggest_HDBSCAN_cluster_UMAP],
    [T6_biggest_HDBSCAN_cluster_TSNE_clustering, T6_biggest_HDBSCAN_cluster_UMAP_clustering],
    ['t-SNE', 'UMAP']):

    labels = clustering.labels_
    noise_mask = labels == -1

    ax.scatter(x=projection[noise_mask, 0],
               y=projection[noise_mask, 1],
               c='lightgrey', s=3, alpha=0.3)
    ax.scatter(x=projection[~noise_mask, 0],
               y=projection[~noise_mask, 1],
               c=labels[~noise_mask], cmap='tab20b', s=3, alpha=0.5)

    ax.set_title(rf"{proj_name} projection of the biggest $\mathbb{{T}}^6$ cluster by HDBSCAN")

plt.suptitle(r"Projections with HDBSCAN clustering", size=20)
plt.tight_layout()

plt.show()

#### On the biggest cluster of the puckers conformations

In [ ]:
def get_biggest_cluster_PC(clust_stock, base):
    
    labels = PC_stock[base].reset_index(drop=True)
    unique, counts = np.unique(labels, return_counts=True)
    largest_class = unique[np.argmax(counts)]
    mask = labels == largest_class
    cluster = clust_stock.reset_index(drop=True)[mask]
    
    return cluster, largest_class

In [ ]:
T6_PC_biggest_cluster_data, T6_PC_biggest_class = get_biggest_cluster_PC(clust_stock = mid_6angles["AAA"], base = "AAA")

In [ ]:
T6_PC_biggest_cluster_TSNE = sm.TSNE(metric=tore_dist_umap).fit_transform(T6_PC_biggest_cluster_data)

In [ ]:
T6_PC_biggest_cluster_UMAP = umap.UMAP(metric=tore_dist_umap).fit_transform(T6_PC_biggest_cluster_data)

In [ ]:
T6_biggest_PC_cluster_TSNE_clustering = HDBSCAN(min_cluster_size=30, metric=tore_dist_umap, copy=True).fit(T6_PC_biggest_cluster_TSNE)

In [ ]:
T6_biggest_PC_cluster_UMAP_clustering = HDBSCAN(min_cluster_size=30, metric=tore_dist_umap, copy=True).fit(T6_PC_biggest_cluster_UMAP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for ax, projection, clustering in zip(
    [axes[0], axes[1]],
    [T6_PC_biggest_cluster_TSNE, T6_PC_biggest_cluster_UMAP],
    [T6_biggest_PC_cluster_TSNE_clustering, T6_biggest_PC_cluster_UMAP_clustering]):

    labels = clustering.labels_
    noise_mask = labels == -1

    ax.scatter(x=projection[noise_mask, 0],
               y=projection[noise_mask, 1],
               c='lightgrey', s=3, alpha=0.3)
    ax.scatter(x=projection[~noise_mask, 0],
               y=projection[~noise_mask, 1],
               c=labels[~noise_mask], cmap='tab20b', s=3, alpha=0.5)

axes[0].set_title(f"t-SNE projection of the biggest PC cluster ({T6_PC_biggest_class})")
axes[1].set_title(f"UMAP projection of the biggest PC cluster ({T6_PC_biggest_class})")

plt.suptitle(f"Projection of the biggest PC cluster : {T6_PC_biggest_class}", size=20)
plt.tight_layout()

plt.show()